## Data Ingestion

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/Annual-Report_Zalando-SE_EN_241203_s.pdf")
pages = loader.load()
print(f"Total pages loaded: {len(pages)}")

/var/folders/sd/yg6lwg754_v_pcl9zl7ttbch0000gn/T/ipykernel_79542/3039257055.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total pages loaded: 276


In [2]:
print(pages[0].page_content[:500])

In [3]:
for i in range(1, 5):  # Example: print metadata for pages 1 through 4
    print(pages[i].page_content[:500])

Key figures
2023 2022 Change
Key performance indicators
Gross Merchandise Volume (GMV*) (in EUR m) 14,631.6 14,788.7  -1.1 %
Revenue (in EUR m) 10,143.1 10,344.8  -1.9 %
Adjusted EBIT (in EUR m) 349.9 184.6  89.5 %
Adjusted EBIT margin (as %) 3.5 1.8 1.7pp
EBIT (in EUR m) 190.9 81.0 >100%
EBIT margin (as %) 1.9 0.8 1.1pp
Capex (in EUR m) -263.2 -351.7  -25.2 %
Active customers (LTM**) (in millions) 49.6 51.2  -3.3 %
Number of orders (in millions) 244.8 261.1  -6.2 %
Average GMV* per active custo
Company
1.1 Foreword 5
1.2 Report of the Supervisory Board 8
1.3 Remuneration report 16
1.4 The Zalando share – 2023 in review 61
Combined management report
2.1 Information on our group 69
2.2 Report on economic position 106
2.3 Risk and opportunity report 119
2.4 Outlook 130
2.5 Corporate governance statement 134
2.6 Takeover law disclosures pursuant to Sections 289a (1), 315a (1) HGB 
and explanatory report
153
2.7 Supplementary management report to the separate financial statements of 
Zalan

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Smarter separators — tries to split at section boundaries first
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # larger chunks = more context per chunk
    chunk_overlap=150,      # more overlap = less meaning lost at boundaries
    separators=[
        "\n\n",             # paragraph breaks first
        "\n",               # then line breaks
        "(?<=\. )",         # then sentence endings
        " ",                # then words
        ""                  # last resort — characters
    ]
)

chunks = splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")
print(f"\n--- First chunk ---")
print(chunks[0].page_content)
print(f"\n--- Second chunk ---")
print(chunks[1].page_content)

Total pages: 276
Total chunks: 864

--- First chunk ---
Key figures
2023 2022 Change
Key performance indicators
Gross Merchandise Volume (GMV*) (in EUR m) 14,631.6 14,788.7  -1.1 %
Revenue (in EUR m) 10,143.1 10,344.8  -1.9 %
Adjusted EBIT (in EUR m) 349.9 184.6  89.5 %
Adjusted EBIT margin (as %) 3.5 1.8 1.7pp
EBIT (in EUR m) 190.9 81.0 >100%
EBIT margin (as %) 1.9 0.8 1.1pp
Capex (in EUR m) -263.2 -351.7  -25.2 %
Active customers (LTM**) (in millions) 49.6 51.2  -3.3 %
Number of orders (in millions) 244.8 261.1  -6.2 %
Average GMV* per active customer (LTM**) (in EUR) 295.2 288.6  2.3 %
Average orders per active customer (LTM**) 4.9 5.1  -3.1 %
Average basket size (LTM**) (in EUR) 59.8 56.6  5.5 %
Other key figures
Net working capital (in EUR m) -441.8 -211.6 >100%
Equity ratio (as % of total assets) 30.5 28.8 1.6pp
Cash flow from operating activities (in EUR m) 949.5 459.9 >100%
Cash flow from investing activities (in EUR m) -320.7 -476.2  32.7 %
Free cash flow (in EUR m) 683.8 -18.

<>:10: SyntaxWarning: invalid escape sequence '\.'
<>:10: SyntaxWarning: invalid escape sequence '\.'
/var/folders/sd/yg6lwg754_v_pcl9zl7ttbch0000gn/T/ipykernel_79542/2432773586.py:10: SyntaxWarning: invalid escape sequence '\.'
  "(?<=\. )",         # then sentence endings


In [5]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")
api_key = os.getenv("GROQ_API_KEY")
print("Key loaded!" if api_key else "Key not found!")

Key loaded!


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [7]:
test = embeddings.embed_query("What was Zalando's revenue in 2023?")
print(f"Vector length: {len(test)}")
print(f"First 5 numbers: {test[:5]}")

Vector length: 384
First 5 numbers: [-0.018109086900949478, 0.06879890710115433, -0.11133615672588348, -0.006607838440686464, 0.006117901299148798]


In [8]:
import shutil
import os

path = os.path.abspath("../vectorstore")

if os.path.exists(path):
    shutil.rmtree(path)

os.makedirs(path, exist_ok=True)

print("Clean vectorstore created at:", path)

Clean vectorstore created at: /Users/pallavijadhav/Projects/AI/FinNews/finnews-rag/vectorstore


In [10]:
# Cell 4 — Build fresh vectorstore
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../vectorstore"
)
print(f"Total chunks stored: {vectorstore._collection.count()}")

Total chunks stored: 864


In [11]:
# Ask a question and see what chunks come back
query = "What were the key risks Zalando faced in 2023?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Page: {doc.metadata['page']}")
    print(doc.page_content)


--- Chunk 1 ---
Page: 125
As an international company, we are exposed to macroeconomic, market and geopolitical 
developments. We deem the external developments described below as especially relevant to 
Zalando as they may influence Zalando’s top risks or lead to new risks in the short-, medium- 
or long-term.
Persistent core inflation, high interest rates and low consumer sentiment were amongst the 
biggest challenges faced by European companies in 2023. This macroeconomic uncertainty is 
not likely to resolve before the end of 2024. The performance of the overall fashion market in 
2024 will depend according to our assessment heavily on the overall macroeconomic 
environment.
Also, there are potential short- and medium-term risks due to global geopolitical instability 
resulting from events like the Russian-Ukraine war or the ongoing conflict in the Middle East. 
Energy markets are particularly sensitive to such risks. If they came to pass, it would add to

--- Chunk 2 ---
Page: 12

In [12]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv("../.env")

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # ← updated model name
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

print("LLM loaded!")

LLM loaded!


In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Build retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Build prompt
prompt = PromptTemplate.from_template("""
Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context: {context}

Question: {question}

Answer:""")

# Helper to format retrieved chunks
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the chain
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Ask your first question!
questions = [
    "What were the key risks Zalando faced in 2023?",
    "How many active customers did Zalando have?",
    "What is Zalando's strategy for future growth?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {chain.invoke(q)}")
    print("---")


Q: What were the key risks Zalando faced in 2023?
A: According to the provided context, the key risks Zalando faced in 2023 were:

1. Risks related to climate change (Material High High)
2. Increasing cyber threats (Significant High Medium High)
3. Market environment and competition (Significant High Medium High)
4. Risks inherent to our business model (New) (Significant High Low)

Additionally, the report mentions that risks related to high inflation and macroeconomic factors were significant in 2023 but have been largely considered in the plan for 2024, with the remaining risk on inflation now assessed as "moderate".
---

Q: How many active customers did Zalando have?
A: Zalando served around 50 million active customers across Europe in 2023.
---

Q: What is Zalando's strategy for future growth?
A: According to the provided context, Zalando's strategy for future growth involves selectively investing through the cycle to create more inspiring, engaging, and entertaining customer expe